Data Source:

The core data source for this project is a list of deaths in National Park units as reported by the US National Park Service in a file downloaded from this page: https://www.nps.gov/aboutus/mortality-data.htm. I'll start with required imports and loading that dataset to a pandas dataframe.

In [372]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import geopandas as gpd

import requests, os, time, lxml, re, unicodedata

from dotenv import load_dotenv

from bs4 import BeautifulSoup

In [373]:
API_KEY = os.getenv("NPS_API_KEY")

print(API_KEY is not None)

True


In [374]:
df = pd.ExcelFile(r"../data/NPS-Mortality-Data-CY2007-to-CY2024-Released-August-2024.xlsx")
df = df.parse(sheet_name="CY2007-Present Q2")
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Outcome,Sex,Age Range,Activity
0,2007-01-01,Glen Canyon National Recreation Area,Undetermined,Undetermined,Undetermined,Fatal injury,Male,65+,Not Reported
1,2007-01-22,Golden Gate National Recreation Area,Drowning,Drowning,Unintentional,Fatal injury,Male,Not Reported,Vessel Related
2,2007-01-22,Golden Gate National Recreation Area,Undetermined,Undetermined,Undetermined,Fatal injury,Male,Not Reported,Vessel Related
3,2007-01-29,Natchez Trace Parkway,Motor Vehicle Crash,Motor Vehicle Crash,Unintentional,Fatal injury,Female,15-24,Driving
4,2007-01-29,Natchez Trace Parkway,Motor Vehicle Crash,Motor Vehicle Crash,Unintentional,Fatal injury,Female,45-54,Driving


I'm going to create a function that will clean the columns that have strings of text.

In [375]:
def clean(col):
    #Verify is string
    col = col.astype("string")

    #Strip leading/trailing whitespace and lowercase
    col = col.str.strip().str.lower()

    #Normalize unicode (remove accents/diacritics: haleakala, honokohau, hawaii, etc.)
    col = col.map(lambda x: unicodedata.normalize("NFKD", x) if pd.notna(x) else x)
    col = col.str.encode("ascii", "ignore").str.decode("ascii")

    #Drop curly apostrophes/okinas
    col = col.str.replace(r"[’ʻ`]", "", regex=True)

    # Normalize different dash types
    col = col.str.replace(r"[-–—]", "-", regex=True)

    # Normalize "&" to "and"
    col = col.str.replace(r"\s*&\s*", " and ", regex=True)

    # Normalize "st." → "st"
    col = col.str.replace(r"\bst\.\s", "st ", regex=True)

    # Collapse multiple spaces to a single space and strip again
    col = col.str.replace(r"\s+", " ", regex=True).str.strip()

    # Treat empty strings as missing
    col = col.replace("", pd.NA)

    return col

In [376]:
df.columns

Index(['Incident Date', 'Park Name', 'Cause of Death',
       'Cause of Death Group \n(Used in the NPS Mortality Dashboard) ',
       'Intent', 'Outcome', 'Sex', 'Age Range', 'Activity'],
      dtype='object')

In [377]:
cols = ["Park Name", 
        "Intent", 
        "Outcome", 
        "Cause of Death", 
        "Cause of Death Group \n(Used in the NPS Mortality Dashboard) ", 
        "Sex", 
        "Activity"]

df[cols] = df[cols].apply(clean)

In [378]:
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Outcome,Sex,Age Range,Activity
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,fatal injury,male,65+,not reported
1,2007-01-22,golden gate national recreation area,drowning,drowning,unintentional,fatal injury,male,Not Reported,vessel related
2,2007-01-22,golden gate national recreation area,undetermined,undetermined,undetermined,fatal injury,male,Not Reported,vessel related
3,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,fatal injury,female,15-24,driving
4,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,fatal injury,female,45-54,driving


In [379]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4635 entries, 0 to 4634
Data columns (total 9 columns):
 #   Column                                                        Non-Null Count  Dtype         
---  ------                                                        --------------  -----         
 0   Incident Date                                                 4635 non-null   datetime64[ns]
 1   Park Name                                                     4635 non-null   object        
 2   Cause of Death                                                4635 non-null   object        
 3   Cause of Death Group 
(Used in the NPS Mortality Dashboard)   4635 non-null   object        
 4   Intent                                                        4635 non-null   object        
 5   Outcome                                                       4635 non-null   object        
 6   Sex                                                           4635 non-null   object        
 7   Age Ran

In [380]:
df.isna().sum()

Incident Date                                                     0
Park Name                                                         0
Cause of Death                                                    0
Cause of Death Group \n(Used in the NPS Mortality Dashboard)      0
Intent                                                            0
Outcome                                                           0
Sex                                                               0
Age Range                                                        12
Activity                                                          0
dtype: int64

I'm going to look more closely at the "Cause of death" and "Cause of Death Group \n..." columns to see if this data is redundant, or if I need to use both in my analysis.

Check if all values in all rows and columns are the same.

In [381]:
(df["Cause of Death"] == df["Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]).all()

np.False_

Since the are not duplicates, I'm going to build a mask to count how many mismatches are in the data.

In [382]:
mask_mismatch = df["Cause of Death"] != df["Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]
print("Total rows:", len(df))
print("Total mismatches:", mask_mismatch.sum())

Total rows: 4635
Total mismatches: 494


In [383]:
df.loc[mask_mismatch, ["Cause of Death", "Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]].head()

,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard)
8,avalanche,environmental
9,vessel incident,other transportation
46,vessel incident,other transportation
48,bicycle only crash,other transportation
56,poisoning - drugs,poisoning


First, I'll focus on "Cause of Death" fields.

In [384]:
df.groupby("Cause of Death").size()

Cause of Death
aircraft incident                          77
altitude                                    1
asphyxiation                                7
avalanche                                  41
bicycle only crash                         23
collapsing earth/sand                       2
drowning                                  864
electrocution                               4
fall                                      505
falling ice                                 2
falling tree/branch                        21
fire/burn                                   5
firearm                                     5
flash flood                                14
homicide                                   51
horseback riding incident                   4
hyperthermia                               83
hypothermia                                46
legal intervention                          2
lightning strike                           10
medical - during physical activity        318
medical - not durin

In [385]:
df["Cause of Death"].nunique()

42

In [386]:
print(sorted(df["Cause of Death"].unique()))

['aircraft incident', 'altitude', 'asphyxiation', 'avalanche', 'bicycle only crash', 'collapsing earth/sand', 'drowning', 'electrocution', 'fall', 'falling ice', 'falling tree/branch', 'fire/burn', 'firearm', 'flash flood', 'homicide', 'horseback riding incident', 'hyperthermia', 'hypothermia', 'legal intervention', 'lightning strike', 'medical - during physical activity', 'medical - not during physical activity', 'medical - unknown', 'motor vehicle crash', 'other', 'poisoning - alcohol', 'poisoning - carbon monoxide', 'poisoning - drugs', 'poisoning - other', 'rockfall', 'skateboard incident', 'skiing incident', 'snowboard incident', 'snowmobile incident', 'struck by/against', 'suicide', 'train crash', 'trolly crash', 'undetermined', 'vessel incident', 'water diving incident', 'widlife incident']


Now I'll look at age range fields.

In [387]:
print(df["Age Range"].unique())

['65+' 'Not Reported' '15-24' '45-54' '25-34' '55-64' '35-44' '0-14' nan
 '45 - 54' '35 - 44' 'Unintentional' '0 - 14']


In [388]:
unknown_age = {"Not Reported", "Unintentional", "Unknown", None, np.nan}

df["age_range_clean"] = df["Age Range"].replace(list(unknown_age), "Unknown")

print(df["age_range_clean"].value_counts())

age_range_clean
65+        827
55-64      676
Unknown    643
45-54      625
25-34      616
15-24      594
35-44      541
0-14       109
45 - 54      2
35 - 44      1
0 - 14       1
Name: count, dtype: int64


In [389]:
df["age_range_clean"] = (
    df["age_range_clean"]
    .str.replace(r"\s*-\s*", "-", regex=True)  # normalize dashes
    .str.strip()
)

In [390]:
print(df["age_range_clean"].value_counts())

age_range_clean
65+        827
55-64      676
Unknown    643
45-54      627
25-34      616
15-24      594
35-44      542
0-14       110
Name: count, dtype: int64


In [391]:
df["Age Range"] = df["age_range_clean"]

df = df.drop(columns=["age_range_clean"])

In [392]:
print(df["Age Range"].unique())

['65+' 'Unknown' '15-24' '45-54' '25-34' '55-64' '35-44' '0-14']


In [393]:
ranges = df["Age Range"].str.extract(r"(?P<min>\d+)-(?P<max>\d+)")
df["Age Range Min"] = ranges["min"].astype(float)
df["Age Range Max"] = ranges["max"].astype(float)

mask_plus = df["Age Range"].str.endswith("+", na=False)
df.loc[mask_plus, "Age Range Min"] = df.loc[mask_plus, "Age Range"].str[:-1].astype(float)
df.loc[mask_plus, "Age Range Max"] = np.nan 

In [394]:
df.dtypes

Incident Date                                                    datetime64[ns]
Park Name                                                                object
Cause of Death                                                           object
Cause of Death Group \n(Used in the NPS Mortality Dashboard)             object
Intent                                                                   object
Outcome                                                                  object
Sex                                                                      object
Age Range                                                                object
Activity                                                                 object
Age Range Min                                                           float64
Age Range Max                                                           float64
dtype: object

In [395]:
print(df[["Age Range","Age Range Min","Age Range Max"]].drop_duplicates())

   Age Range  Age Range Min  Age Range Max
0        65+           65.0            NaN
1    Unknown            NaN            NaN
3      15-24           15.0           24.0
4      45-54           45.0           54.0
7      25-34           25.0           34.0
10     55-64           55.0           64.0
11     35-44           35.0           44.0
13      0-14            0.0           14.0


Now I'll look at the outcome column.

In [396]:
df["Outcome"].unique()

array(['fatal injury'], dtype=object)

This column doesn't add anything. We know each event was a fatal injury. I'm going to drop the entire column.

In [397]:
df.drop("Outcome", axis=1, inplace=True)

In [398]:
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Sex,Age Range,Activity,Age Range Min,Age Range Max
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,65.0,NaN
1,2007-01-22,golden gate national recreation area,drowning,drowning,unintentional,male,Unknown,vessel related,NaN,NaN
2,2007-01-22,golden gate national recreation area,undetermined,undetermined,undetermined,male,Unknown,vessel related,NaN,NaN
3,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,15-24,driving,15.0,24.0
4,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,45-54,driving,45.0,54.0


Now I'll look at "Intent"

In [399]:
df["Intent"].value_counts()

Intent
unintentional    2713
intentional       701
medical           640
undetermined      581
Name: count, dtype: int64

I want to see if the "Medical" rows all correspond with the various medical causes of death. I'm going to normalize the Intent and Cause of Death columns and then compare to how many I would expect with the medical categories (3).

In [400]:
#See what causes appear under Intent == "Medical"
medical_causes_found = (
    df.loc[df["Intent"] == "medical", "Cause of Death"]
      .dropna()
      .unique()
)
print(medical_causes_found)

['medical - not during physical activity'
 'medical - during physical activity' 'medical - unknown']


This is what I expected. I'm just going to double check that the number of medical intent fields matches the sum of the three types of medical cause fields.

In [401]:
medical_causes = [
    "medical - not during physical activity",
    "medical - during physical activity",
    "medical - unknown"
]

df["Cause of Death"].value_counts().loc[medical_causes]

Cause of Death
medical - not during physical activity    240
medical - during physical activity        318
medical - unknown                          82
Name: count, dtype: int64

There are 640 medical causes, and 640 medical intents, so I believe they corrospond correctly. 

In [402]:
df["Sex"].value_counts()

Sex
male            3484
female           856
not reported     295
Name: count, dtype: int64

These are the values we would expect and there are no missing values. No additional work is necessary.

In [403]:
df["Activity"].head()

0      not reported
1    vessel related
2    vessel related
3           driving
4           driving
Name: Activity, dtype: object

In [404]:
df["Activity"].value_counts()

Activity
driving                       837
not reported                  723
suicide                       650
hiking                        502
swimming                      447
vessel related                341
other                         198
climbing                      149
walking                       123
flying                         78
illegal activity               65
bicycling                      64
homicide                       51
sleeping                       46
fishing                        42
camping                        42
snorkeling                     36
sitting                        31
photographing                  27
skiing                         23
diving (land)                  23
wading                         21
canyoneering                   20
playing sports                 20
diving - scuba                 15
base jumping                    8
eating                          7
crossing river                  7
hunting                         6
rock 

In [405]:
df["Activity"].sort_values().unique()

array(['base jumping', 'bicycling', 'camping', 'canyoneering', 'caving',
       'climbing', 'crossing river', 'diving (land)', 'diving - free',
       'diving - scuba', 'driving', 'eating', 'fishing', 'flying',
       'hiking', 'homicide', 'horseback (or mule) riding', 'hunting',
       'illegal activity', 'not reported', 'other', 'photographing',
       'playing sports', 'rock scrambeling', 'rock scrambling', 'sitting',
       'skateboarding', 'skiing', 'sleeping', 'snorkeling',
       'snowboarding', 'snowmobiling', 'snowshoeing', 'suicide',
       'swimming', 'undetermined', 'vessel related', 'wading', 'walking',
       'wildlife watching'], dtype=object)

I can combine "Rock Scrambeling" and "Rock Scrambling". I considered combining "Not Reported" and "Undetermined, but chose not to at this time because they do convey different information.

In [406]:
df["Activity"] = df["Activity"].replace(
    "Rock Scrambeling", "Rock Scrambling"
)

df["Activity"].sort_values().unique()

array(['base jumping', 'bicycling', 'camping', 'canyoneering', 'caving',
       'climbing', 'crossing river', 'diving (land)', 'diving - free',
       'diving - scuba', 'driving', 'eating', 'fishing', 'flying',
       'hiking', 'homicide', 'horseback (or mule) riding', 'hunting',
       'illegal activity', 'not reported', 'other', 'photographing',
       'playing sports', 'rock scrambeling', 'rock scrambling', 'sitting',
       'skateboarding', 'skiing', 'sleeping', 'snorkeling',
       'snowboarding', 'snowmobiling', 'snowshoeing', 'suicide',
       'swimming', 'undetermined', 'vessel related', 'wading', 'walking',
       'wildlife watching'], dtype=object)

In [407]:
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Sex,Age Range,Activity,Age Range Min,Age Range Max
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,65.0,NaN
1,2007-01-22,golden gate national recreation area,drowning,drowning,unintentional,male,Unknown,vessel related,NaN,NaN
2,2007-01-22,golden gate national recreation area,undetermined,undetermined,undetermined,male,Unknown,vessel related,NaN,NaN
3,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,15-24,driving,15.0,24.0
4,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,45-54,driving,45.0,54.0


In [408]:
df.columns

Index(['Incident Date', 'Park Name', 'Cause of Death',
       'Cause of Death Group \n(Used in the NPS Mortality Dashboard) ',
       'Intent', 'Sex', 'Age Range', 'Activity', 'Age Range Min',
       'Age Range Max'],
      dtype='object')

In [409]:
df = df.rename(columns={"Cause of Death Group \n(Used in the NPS Mortality Dashboard) ": "Cause Category"})

df.columns

Index(['Incident Date', 'Park Name', 'Cause of Death', 'Cause Category',
       'Intent', 'Sex', 'Age Range', 'Activity', 'Age Range Min',
       'Age Range Max'],
      dtype='object')

I know that NPS visitation data is available in monthly increments only, so I'm going to pull the month and year out of the datetime field to create new columns I can use when I compare visitation data.

In [410]:
df["Month"] = df["Incident Date"].dt.month
df["Year"]  = df["Incident Date"].dt.year
df.head(1)

,Incident Date,Park Name,Cause of Death,Cause Category,Intent,Sex,Age Range,Activity,Age Range Min,Age Range Max,Month,Year
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,65.0,NaN,1,2007


Unfortunately, there are several unique visit park codes assigned to some parks. The park code is a key required to interact with may NPS APIs. There is not a complete listing of park codes available but I know I want to obtain the full visitation data set, so I've found a drop down list with all the codes used by this dataset. I used the inspect panel in a Windows browser to copy the entries from the bounded list, which is saved in my data folder as visit_codes.html. You can obtain updated information using the same process at https://irma.nps.gov/Stats/Reports/Park

In [411]:
with open("../data/visit_codes.html", "r", encoding="utf-8") as f:
    html = f.read()

print(len(html))
print(html[:500])

88925
<li role="option" unselectable="on" class="x-boundlist-item" tabindex="-1" data-recordindex="0" data-recordid="1" data-boundview="filtercombobox-1012-picker" aria-selected="false">Abraham Lincoln Birthplace NHP (ABLI)</li><li role="option" unselectable="on" class="x-boundlist-item" tabindex="-1" data-recordindex="1" data-recordid="2" data-boundview="filtercombobox-1012-picker" aria-selected="false">Acadia NP (ACAD)</li><li role="option" unselectable="on" class="x-boundlist-item" tabindex="-1" da


Now I'm going to use BeautifulSoup to find all the items in the boundlist that I copied. This should be all my parks. I'm going to look at the length to see if it's working correctly. Each item that has a class="x-boundlist-item" should be an individual park, so I'm going to divide those into a list and see how many there are. I'm expcting just a little over 400, because that's how many national park units I know exist.

In [412]:
soup = BeautifulSoup(html, "html.parser")

items = soup.find_all("li", class_="x-boundlist-item")
print(len(items))
print(items[0:5])

415
[<li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="1" data-recordindex="0" role="option" tabindex="-1" unselectable="on">Abraham Lincoln Birthplace NHP (ABLI)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="2" data-recordindex="1" role="option" tabindex="-1" unselectable="on">Acadia NP (ACAD)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="3" data-recordindex="2" role="option" tabindex="-1" unselectable="on">Adams NHP (ADAM)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="4" data-recordindex="3" role="option" tabindex="-1" unselectable="on">African Burial Ground NM (AFBG)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="5" data-recordindex="4" role="option"

Now that it's divided into each list item instead of a giant string of code, I'm going to put it into rows.

In [413]:
#create blank place to hold the rows when they're created
rows = []

#create for loop that will go through all of the boundlist items 
for li in items:
    #get just the text that would display withough the rest of the code
    text = li.get_text(strip=True)   

    #I'm going to use a regular expression to seperate the part in parentheses from the rest
    m = re.match(r"^(.*)\(([^)]+)\)$", text)
    if m:
        name = m.group(1).strip()
        code = m.group(2).strip()
        rows.append({"UnitName": name, "UnitCode": code})
    else:
        # in case some weird row doesn't match that pattern
        rows.append({"UnitName": text, "UnitCode": None})


In [414]:
df_visit_codes = pd.DataFrame(rows)

df_visit_codes.head()

,UnitName,UnitCode
0,Abraham Lincoln Birthplace NHP,ABLI
1,Acadia NP,ACAD
2,Adams NHP,ADAM
3,African Burial Ground NM,AFBG
4,Agate Fossil Beds NM,AGFO


In [415]:
df_visit_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   UnitName  415 non-null    object
 1   UnitCode  415 non-null    object
dtypes: object(2)
memory usage: 6.6+ KB


In [416]:
df_visit_codes = df_visit_codes.rename(
    columns={"UnitCode": "Visit Code",
    "UnitName": "Visit Name"})

In [417]:
cols = ["Visit Name", 
        "Visit Code"]

df_visit_codes[cols] = df_visit_codes[cols].apply(clean)

In [418]:
df_visit_codes.head(1)

,Visit Name,Visit Code
0,abraham lincoln birthplace nhp,abli


I'll use the API to call for all the parks available in the list and obtain all available visitor data, which I will then map to my dataset.

In [419]:
BASE = "https://irmaservices.nps.gov/v3/rest/stats/visitation"

all_codes = (
    df_visit_codes["Visit Code"]
    .dropna()
    .astype(str)
    .str.upper()
    .unique()
    .tolist()
)

def chunk_list(lst, n):
    #Yield successive n-sized chunks from lst.
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

CODES_PER_CALL = 50

all_results = []

with requests.Session() as s:
    for chunk in chunk_list(all_codes, CODES_PER_CALL):
        params = {
            "unitCodes": ",".join(chunk),
            "startMonth": 1,
            "startYear": 2007,
            "endMonth": 6,
            "endYear": 2024,
            "format": "json",
        }
        r = s.get(BASE, params=params, headers={"Accept": "application/json"}, timeout=30)
        r.raise_for_status()
        data = r.json()
        all_results.extend(data)

df_visits = pd.DataFrame(all_results)

In [420]:
df_visits.head()

,Month,NonRecreationVisitors,RecreationVisitors,UnitCode,UnitName,Year
0,1,0,4960,ABLI,Abraham Lincoln Birthplace NHP,2007
1,2,0,5877,ABLI,Abraham Lincoln Birthplace NHP,2007
2,3,0,9868,ABLI,Abraham Lincoln Birthplace NHP,2007
3,4,0,17900,ABLI,Abraham Lincoln Birthplace NHP,2007
4,5,0,21277,ABLI,Abraham Lincoln Birthplace NHP,2007


In [421]:
df_visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78652 entries, 0 to 78651
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Month                  78652 non-null  int64 
 1   NonRecreationVisitors  78652 non-null  int64 
 2   RecreationVisitors     78652 non-null  int64 
 3   UnitCode               78652 non-null  object
 4   UnitName               78652 non-null  object
 5   Year                   78652 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 3.6+ MB


In [422]:
df_visits = df_visits.rename(
    columns = {"RecreationVisitors": "Recreation Visitors",
               "NonRecreationVisitors": "Others",
               "UnitCode": "Visit Code",
               "UnitName": "Visit Name"}
)

df_visits.head(1)

,Month,Others,Recreation Visitors,Visit Code,Visit Name,Year
0,1,0,4960,ABLI,Abraham Lincoln Birthplace NHP,2007


In [423]:
df_visits.to_csv("../data/NPS_visitation_data.csv", index=False)

In [424]:
df_visits["Visit Code"].value_counts().head()

Visit Code
ZION    210
ABLI    210
ACAD    210
ADAM    210
YUCH    210
Name: count, dtype: int64

In [425]:
df_visits["Visit Code"].value_counts().tail()

Visit Code
FRST    18
STGE     6
TILL     6
IATR     6
AMCH     6
Name: count, dtype: int64

In [426]:
counts = df_visits["Visit Code"].value_counts()

len(counts[counts != 210])

38

Since we are missing some parks entirely and some don't have complete data, I'm going to do a couple of sanity checks. First, I'll call the API for one of the unit codes that didn't have any data, and then for one with incomplete data. I want to ensure the data is actually missing and there wasn't a problem with my API call.

In [427]:
BASE = "https://irmaservices.nps.gov/v3/rest/stats/visitation"
params = {
    "unitCodes": "AMCH",
    "startMonth": 1, "startYear": 2007,
    "endMonth": 6,   "endYear": 2024,
    "format": "json",
}

r = requests.get(BASE, params=params, headers={"Accept": "application/json"}, timeout=30)

In [428]:
AMCH = r.json()
len(AMCH)

6

In [429]:
missing_visits = set(df_visit_codes["Visit Code"]) - set(df_visits["Visit Code"])
missing_visits

{'abli',
 'acad',
 'adam',
 'afbg',
 'agfo',
 'alag',
 'alfl',
 'alpo',
 'amch',
 'amis',
 'ande',
 'ania',
 'anjo',
 'anti',
 'apco',
 'apis',
 'arch',
 'arho',
 'arpo',
 'asis',
 'azru',
 'badl',
 'band',
 'bela',
 'beol',
 'bepa',
 'bibe',
 'bica',
 'bicr',
 'bicy',
 'biho',
 'bisc',
 'biso',
 'bith',
 'blca',
 'blri',
 'blrv',
 'blue',
 'boaf',
 'boha',
 'bost',
 'bowa',
 'brca',
 'brcr',
 'brvb',
 'buff',
 'buis',
 'cabr',
 'cach',
 'cacl',
 'caco',
 'cagr',
 'caha',
 'cakr',
 'calo',
 'camo',
 'cana',
 'cane',
 'cany',
 'care',
 'cari',
 'carl',
 'casa',
 'cato',
 'cave',
 'cavo',
 'cawo',
 'cebe',
 'cebr',
 'cech',
 'cham',
 'chat',
 'chch',
 'chcu',
 'chic',
 'chir',
 'chis',
 'choh',
 'chpi',
 'chri',
 'chsc',
 'chyo',
 'ciro',
 'clba',
 'colm',
 'colo',
 'cong',
 'coro',
 'cowp',
 'crla',
 'crmo',
 'cuga',
 'cuis',
 'cure',
 'cuva',
 'daav',
 'ddem',
 'dena',
 'depo',
 'deso',
 'deto',
 'deva',
 'dewa',
 'dino',
 'drto',
 'ebla',
 'edal',
 'edis',
 'efmo',
 'eise',
 'elma',
 

In [430]:
BASE = "https://irmaservices.nps.gov/v3/rest/stats/visitation"
params = {
    "unitCodes": "BICR",
    "startMonth": 1, "startYear": 2007,
    "endMonth": 6,   "endYear": 2024,
    "format": "json",
}

r = requests.get(BASE, params=params, headers={"Accept": "application/json"}, timeout=30)

BICR = r.json()
len(BICR)

0

The data appears to be truly missing.

To obtain the rest of the data, I need to compile a master list of all park codes that could be found. I'll use this both to build the API query and to merge obtained datasets. I'll start with the current, official unit list. The list is saved in the data folder of this project as NPS-Unit-List.xlxs, and can be obtained by downloading from this page: https://www.nps.gov/aboutus/foia/foia-reading-room.htm

In [431]:
df_units = pd.read_excel("../data/NPS-Unit-List.xlsx")

df_units.head()

,Name,Type,Park Code,Org Code,Region
0,NPS unit type designations (link),NaN,NaN,NaN,NaN
1,ABRAHAM LINCOLN BIRTHPLACE,NHP,ABLI,5540,SER
2,ACADIA,NP,ACAD,1700,NER
3,ADAMS,NHP,ADAM,1710,NER
4,AFRICAN BURIAL GROUND,NM,AFBG,1762,NER


In [432]:
df_units.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 428 entries, 0 to 427
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Name       428 non-null    object
 1   Type       424 non-null    object
 2   Park Code  425 non-null    object
 3   Org Code   368 non-null    object
 4   Region     425 non-null    object
dtypes: object(5)
memory usage: 16.8+ KB


In [433]:
df_units[df_units["Park Code"].isna()]["Name"]

0      NPS unit type designations (link)
262              MARTIN LUTHER KING, JR.
417                          WORLD WAR I
Name: Name, dtype: object

In [434]:
df_units = df_units.drop(df_units.index[0])

df_units.head()

,Name,Type,Park Code,Org Code,Region
1,ABRAHAM LINCOLN BIRTHPLACE,NHP,ABLI,5540,SER
2,ACADIA,NP,ACAD,1700,NER
3,ADAMS,NHP,ADAM,1710,NER
4,AFRICAN BURIAL GROUND,NM,AFBG,1762,NER
5,AGATE FOSSIL BEDS,NM,AGFO,6710,MWR


In [435]:
cols = ["Name", 
        "Type", 
        "Park Code", 
        "Org Code", 
        "Region"]

df_units[cols] = df_units[cols].apply(clean)

df_units.head()

,Name,Type,Park Code,Org Code,Region
1,abraham lincoln birthplace,nhp,abli,5540,ser
2,acadia,np,acad,1700,ner
3,adams,nhp,adam,1710,ner
4,african burial ground,nm,afbg,1762,ner
5,agate fossil beds,nm,agfo,6710,mwr


In [436]:
df_units = df_units.rename(columns={
    "Name": "Official Park Name",
    "Park Code": "Official Park Code"
})

df_units.head(1)

,Official Park Name,Type,Official Park Code,Org Code,Region
1,abraham lincoln birthplace,nhp,abli,5540,ser


Next, I'll obtain the full list of park units from the parks API. This API does require a key, which can obtained by registering through this link: https://www.nps.gov/subjects/developer/guides.htm

In [437]:
def fetch_all_parks(api_key: str, page_size: int = 50, pause: float = 0.2) -> pd.DataFrame:
    base = "https://developer.nps.gov/api/v1/parks"
    start = 0
    rows = []

#Create a loop that will continue until there is no data or the data returned is < the page size
    while True:
        params = {"limit": page_size, "start": start, "api_key": api_key}
        resp = requests.get(base, params=params, timeout=30)
        resp.raise_for_status()
        #convert the response from JSON text to Python dictionary
        payload = resp.json()

        data = payload.get("data", [])
        if not data:
            break
        
        #The default parks response has more info than I need, so I'm going to filter just for the name and park code
        for item in data:
            rows.append({
                "API Full Name": item.get("fullName"),
                "API Park Code": item.get("parkCode")
            })

        if len(data) < page_size:
            break

        start += page_size
        time.sleep(pause)

    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

df_api_codes = fetch_all_parks(API_KEY, page_size=50)

df_api_codes.head()

,API Full Name,API Park Code
0,Abraham Lincoln Birthplace National Historical...,abli
1,Acadia National Park,acad
2,Adams National Historical Park,adam
3,African American Civil War Memorial,afam
4,African Burial Ground National Monument,afbg


In [438]:
cols = ["API Full Name", 
        "API Park Code"]

df_api_codes[cols] = df_api_codes[cols].apply(clean)

df_api_codes.head()

,API Full Name,API Park Code
0,abraham lincoln birthplace national historical...,abli
1,acadia national park,acad
2,adams national historical park,adam
3,african american civil war memorial,afam
4,african burial ground national monument,afbg


Because park codes have changed overtime and records may not necessarily be updated, I will also use a historic listing of codes to ensure I am able to obtain all the data I need. I will scrape this information from the table shown at https://www.nps.gov/articles/000/historic-listing-of-nps-park-codes.htm

In [439]:
url = "https://www.nps.gov/articles/000/historic-listing-of-nps-park-codes.htm"
tables = pd.read_html(url)
print(len(tables))          # how many tables pandas found

1


In [440]:
df_historic_codes = tables[0]
df_historic_codes.head()

,0,1
0,"Park, Program, or Office Name",Alpha Code
1,Abraham Lincoln Birthplace National Historical...,ABLI
2,Abraham Lincoln National Heritage Area,See MWR
3,Acadia National Park,ACAD
4,"Adams Memorial, District of Columbia",See NCR


In [441]:
df_historic_codes.columns = df_historic_codes.iloc[0]
df_historic_codes = df_historic_codes.drop(df_historic_codes.index[0])
df_historic_codes = df_historic_codes.reset_index(drop=True)
df_historic_codes.columns = df_historic_codes.columns.str.strip()
df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code
0,Abraham Lincoln Birthplace National Historical...,ABLI
1,Abraham Lincoln National Heritage Area,See MWR
2,Acadia National Park,ACAD
3,"Adams Memorial, District of Columbia",See NCR
4,Adams National Historical Park,ADAM


In [442]:
df_historic_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Park, Program, or Office Name  614 non-null    object
 1   Alpha Code                     614 non-null    object
dtypes: object(2)
memory usage: 9.7+ KB


I need to clean up the Alpha Code column based on what I can see on the site. First, I'll remove "See " from the beginning of rows that are alternate or historic names. I'll start by creating a new column called "Alternate Name" with a boolean value that designates those rows, and then drop the text "See" from the "Alpha Code" column.

In [443]:
df_historic_codes["Alternate Name"] = (
    df_historic_codes["Alpha Code"].str.startswith("See ")
)

df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code,Alternate Name
0,Abraham Lincoln Birthplace National Historical...,ABLI,False
1,Abraham Lincoln National Heritage Area,See MWR,True
2,Acadia National Park,ACAD,False
3,"Adams Memorial, District of Columbia",See NCR,True
4,Adams National Historical Park,ADAM,False


In [444]:
df_historic_codes["Alpha Code"] = (
    df_historic_codes["Alpha Code"].str.replace(r"^See\s+", "", regex=True)
)

df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code,Alternate Name
0,Abraham Lincoln Birthplace National Historical...,ABLI,False
1,Abraham Lincoln National Heritage Area,MWR,True
2,Acadia National Park,ACAD,False
3,"Adams Memorial, District of Columbia",NCR,True
4,Adams National Historical Park,ADAM,False


Next, I'll drop all rows that have a 3 character park code, as these reference regions instead of individual sites. I am not working with this data on a regional level, so they aren't needed.

In [445]:
df_historic_codes = df_historic_codes[~(df_historic_codes["Alpha Code"].str.len() == 3)]

df_historic_codes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 589 entries, 0 to 613
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Park, Program, or Office Name  589 non-null    object
 1   Alpha Code                     589 non-null    object
 2   Alternate Name                 589 non-null    bool  
dtypes: bool(1), object(2)
memory usage: 14.4+ KB


In [446]:
cols = ["Park, Program, or Office Name", 
        "Alpha Code"]

df_historic_codes[cols] = df_historic_codes[cols].apply(clean)

df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code,Alternate Name
0,abraham lincoln birthplace national historical...,abli,False
2,acadia national park,acad,False
4,adams national historical park,adam,False
5,african american civil war memorial,afam,False
6,african burial ground national monument,afbg,False


In [447]:
df_historic_codes = df_historic_codes.rename(columns={
    "Park, Program, or Office Name": "Historic Park Name",
    "Alpha Code": "Historic Park Code"
})

df_historic_codes.head(1)

,Historic Park Name,Historic Park Code,Alternate Name
0,abraham lincoln birthplace national historical...,abli,False


Now that I've obtained codes from every source I can find, I am going to write a function that creates a list of all available codes. I'll use this to query the remaining APIs.

In [448]:
def get_all_park_codes(df_historic_codes, df_api_codes, df_units, df_visits):

    historic = df_historic_codes["Historic Park Code"].dropna().str.lower().unique()
    api      = df_api_codes["API Park Code"].dropna().str.lower().unique()
    official = df_units["Official Park Code"].dropna().str.lower().unique()
    visits   = df_visits["Visit Code"].dropna().str.lower().unique()

    return sorted(set(historic) | set(api) | set(official) | set(visits))


In [449]:
all_codes = get_all_park_codes(
    df_historic_codes,
    df_api_codes,
    df_units,
    df_visits
)

len(all_codes), all_codes[:5]

(594, ['abli', 'acad', 'adam', 'afam', 'afbg'])

In [450]:
df.head(1)

,Incident Date,Park Name,Cause of Death,Cause Category,Intent,Sex,Age Range,Activity,Age Range Min,Age Range Max,Month,Year
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,65.0,NaN,1,2007


In [451]:
df_visit_codes

,Visit Name,Visit Code
0,abraham lincoln birthplace nhp,abli
1,acadia np,acad
2,adams nhp,adam
3,african burial ground nm,afbg
4,agate fossil beds nm,agfo
...,...,...
410,wupatki nm,wupa
411,yellowstone np,yell
412,yosemite np,yose
413,yukon-charley rivers npres,yuch


In [452]:
def add_park_codes(
    df,
    df_historic_codes,
    df_api_codes,
    df_units,
    df_visits,
):
    out = df.copy()

    # --- Historic Code ---
    hist_map = (
        df_historic_codes
        .dropna(subset=["Historic Park Name", "Historic Park Code"])
        .drop_duplicates(subset="Historic Park Name")
        .set_index("Historic Park Name")["Historic Park Code"]
    )
    out["Historic Park Code"] = out["Park Name"].map(hist_map)

    # --- API Code ---
    api_map = (
        df_api_codes
        .dropna(subset=["API Full Name", "API Park Code"])
        .drop_duplicates(subset="API Full Name")
        .set_index("API Full Name")["API Park Code"]
    )
    out["API Park Code"] = out["Park Name"].map(api_map)

    # --- Official Code ---
    official_map = (
        df_units
        .dropna(subset=["Official Park Name", "Official Park Code"])
        .drop_duplicates(subset="Official Park Name")
        .set_index("Official Park Name")["Official Park Code"]
    )
    out["Official Park Code"] = out["Park Name"].map(official_map)

    # --- Visit Code ---
    visit_map = (
        df_visits
        .dropna(subset=["Visit Name", "Visit Code"])
        .drop_duplicates(subset="Visit Name")
        .set_index("Visit Name")["Visit Code"]
    )
    out["Visit Code"] = out["Park Name"].map(visit_map)

    return out


In [453]:
df_with_codes = add_park_codes(
    df,
    df_historic_codes,
    df_api_codes,
    df_units,
    df_visits,
)

df_with_codes.head()

,Incident Date,Park Name,Cause of Death,Cause Category,Intent,Sex,Age Range,Activity,Age Range Min,Age Range Max,Month,Year,Historic Park Code,API Park Code,Official Park Code,Visit Code
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,65.0,NaN,1,2007,glca,glca,NaN,NaN
1,2007-01-22,golden gate national recreation area,drowning,drowning,unintentional,male,Unknown,vessel related,NaN,NaN,1,2007,goga,goga,NaN,NaN
2,2007-01-22,golden gate national recreation area,undetermined,undetermined,undetermined,male,Unknown,vessel related,NaN,NaN,1,2007,goga,goga,NaN,NaN
3,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,15-24,driving,15.0,24.0,1,2007,natr,natr,natr,NaN
4,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,45-54,driving,45.0,54.0,1,2007,natr,natr,natr,NaN


In [454]:
df_with_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4635 entries, 0 to 4634
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Incident Date       4635 non-null   datetime64[ns]
 1   Park Name           4635 non-null   object        
 2   Cause of Death      4635 non-null   object        
 3   Cause Category      4635 non-null   object        
 4   Intent              4635 non-null   object        
 5   Sex                 4635 non-null   object        
 6   Age Range           4635 non-null   object        
 7   Activity            4635 non-null   object        
 8   Age Range Min       3992 non-null   float64       
 9   Age Range Max       3165 non-null   float64       
 10  Month               4635 non-null   int32         
 11  Year                4635 non-null   int32         
 12  Historic Park Code  4375 non-null   object        
 13  API Park Code       4491 non-null   object      

In [455]:
missing_api_parks = (
    df_with_codes[df_with_codes["API Park Code"].isna()]["Park Name"]
    .dropna()
    .unique()
)
missing_api_parks

array(['new river gorge national river', 'suitland',
       'george washington birthplace',
       'wrangell - st elias national park and reserve',
       "president's park (white house)", 'not reported'], dtype=object)

In [456]:
def get_parks_missing_codes(df_with_codes):

    code_columns = {
        "historic": "Historic Park Code",
        "api": "API Park Code",
        "official": "Official Park Code",
        "visit": "Visit Code"
    }

    missing = {}

    for label, col in code_columns.items():
        parks = (
            df_with_codes[df_with_codes[col].isna()]["Park Name"]
            .dropna()
            .unique()
        )
        missing[label] = sorted(parks)

    return missing


In [457]:
missing_parks = get_parks_missing_codes(df_with_codes)

missing_parks["api"]

['george washington birthplace',
 'new river gorge national river',
 'not reported',
 "president's park (white house)",
 'suitland',
 'wrangell - st elias national park and reserve']

In [470]:
missing_all = df_with_codes[
    df_with_codes["API Park Code"].isna()
    & df_with_codes["Historic Park Code"].isna()
    & df_with_codes["Official Park Code"].isna()
    & df_with_codes["Visit Code"].isna()
]

In [459]:
missing_all_parks = missing_all["Park Name"].unique()
missing_all_parks


array(['suitland', 'wrangell - st elias national park and reserve',
       'not reported'], dtype=object)

I will not be able to match Suitland, as there are several potential park units that may be refered to as Suitland. The other missing item - "wrangell - st elias national park and reserve" is actully a typo. I'll corect that and then there should be a direct match.

In [468]:
#Correct name in base df
df["Park Name"] = df["Park Name"].replace(
    "wrangell - st elias national park and reserve",
    "wrangell - st elias national park and preserve"
)

#Rebuild df_with_codes
df_with_codes = add_park_codes(
    df,
    df_historic_codes,
    df_api_codes,
    df_units,
    df_visits,
)

mask = df_with_codes["Park Name"] == "wrangell - st elias national park and preserve"

df_with_codes.loc[mask, "Visit Code"] = "WRST"

In [472]:
missing_all = df_with_codes[
    df_with_codes["API Park Code"].isna()
    & df_with_codes["Historic Park Code"].isna()
    & df_with_codes["Official Park Code"].isna()
    & df_with_codes["Visit Code"].isna()
]

missing_all_parks = missing_all["Park Name"].unique()
missing_all_parks

array(['suitland', 'not reported'], dtype=object)